In [53]:
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns
import pandas as pd
from PIL import Image
import json
from pathlib import Path
from tqdm import tqdm
import shutil

In [57]:
root = Path(r"D:/Project/CPV_paper/datasets/raw/TerraIncognita")
ann_dir = root / "eccv_18_annotation_files"
img_dir = root / "eccv_18_all_images_sm"

split_files = {
    "train": "train_annotations.json",
    "cis_val": "cis_val_annotations.json",
    "cis_test": "cis_test_annotations.json",
    "trans_val": "trans_val_annotations.json",
    "trans_test": "trans_test_annotations.json",
}

rows = []

for split, filename in split_files.items():
    with open(ann_dir / filename, "r") as f:
        data = json.load(f)

    image_dict = {img["id"]: img for img in data["images"]}
    category_dict = {cat["id"]: cat["name"] for cat in data["categories"]}

    for ann in data["annotations"]:
        img = image_dict[ann["image_id"]]

        rows.append({
            "original_split": split,
            "image_id": img["id"],
            "file_name": img["file_name"],
            "path": str(img_dir / img["file_name"]),
            "location": img["location"],
            "domain": f"L{img['location']}",
            "seq_id": img.get("seq_id"),
            "seq_num_frames": img.get("seq_num_frames"),
            "frame_num": img.get("frame_num"),
            "category_id": ann["category_id"],
            "class_name": category_dict[ann["category_id"]],
        })

df = pd.DataFrame(rows)

In [58]:
domainbed_domains = ["L100", "L38", "L43", "L46"]

df = df[df["domain"].isin(domainbed_domains)].copy()

In [59]:
domainbed_classes = [
    "bird",
    "bobcat",
    "cat",
    "coyote",
    "dog",
    "empty",
    "opossum",
    "rabbit",
    "raccoon",
    "squirrel",
]

df = df[df["class_name"].isin(domainbed_classes)].copy()
df["label"] = pd.Categorical(df["class_name"]).codes

In [65]:
df[df['domain'] == "L43"]['original_split'].unique()

array(['train', 'cis_val', 'cis_test'], dtype=object)

In [54]:
# Better label mapping: keeps label order exactly the same as domainbed_classes
class_to_label = {class_name: i for i, class_name in enumerate(domainbed_classes)}

df = df[df["class_name"].isin(domainbed_classes)].copy()
df["label"] = df["class_name"].map(class_to_label)

# Keep only DomainBed TerraIncognita domains
domainbed_domains = ["L100", "L38", "L43", "L46"]
df = df[df["domain"].isin(domainbed_domains)].copy()

# Original TerraIncognita split -> your processed split
split_map = {
    "train": "train",
    "cis_val": "test",
    "cis_test": "test",
    "trans_val": "test",
    "trans_test": "test",
}

df["new_split"] = df["original_split"].map(split_map)

# Remove anything that did not map correctly
df = df[df["new_split"].notna()].copy()

processed_root = Path(r"D:\Project\CPV_paper\datasets\processed\TerranIncognita")

# Create folders:
# TerranIncognita/L100/train/bird
# TerranIncognita/L100/test/bird
# ...
for domain in domainbed_domains:
    for split in ["train", "test"]:
        for class_name in domainbed_classes:
            folder = processed_root / domain / split / class_name
            folder.mkdir(parents=True, exist_ok=True)

# Copy images
copied = 0
missing = 0

for _, row in tqdm(df.iterrows(), total=len(df)):
    src_path = Path(row["path"])

    domain = row["domain"]
    split = row["new_split"]
    class_name = row["class_name"]

    # Use image_id to avoid duplicate filename problems
    dst_filename = f"{row['image_id']}_{src_path.name}"
    dst_path = processed_root / domain / split / class_name / dst_filename

    if not src_path.exists():
        print("Missing:", src_path)
        missing += 1
        continue

    shutil.copy2(src_path, dst_path)
    copied += 1

print("Done.")
print("Copied:", copied)
print("Missing:", missing)

100%|██████████| 25419/25419 [00:37<00:00, 679.08it/s]

Done.
Copied: 25419
Missing: 0


In [55]:
summary = []

for domain in domainbed_domains:
    for split in ["train", "test"]:
        for class_name in domainbed_classes:
            folder = processed_root / domain / split / class_name
            count = len(list(folder.glob("*")))

            summary.append({
                "domain": domain,
                "split": split,
                "class_name": class_name,
                "count": count,
            })

summary_df = pd.DataFrame(summary)
summary_df

,domain,split,class_name,count
0,L100,train,bird,0
1,L100,train,bobcat,0
2,L100,train,cat,0
3,L100,train,coyote,0
4,L100,train,dog,0
...,...,...,...,...
75,L46,test,empty,726
76,L46,test,opossum,1339
77,L46,test,rabbit,566
78,L46,test,raccoon,1464


In [56]:
summary_df.groupby(["domain", "split"])["count"].sum()

domain  split
L100    test     4879
        train       0
L38     test     5720
        train    4047
L43     test     2846
        train    1174
L46     test     6122
        train       0
Name: count, dtype: int64